# MoodFit V3 - Street class cap comparison

This notebook keeps the V2 model and fine-tuning setup, but caps the street class in the training split at 10,000 images.

- The validation split is unchanged.
- All non-street training images are kept.
- Training augmentation matches V2.
- Epoch logs and the best Macro F1 checkpoint are saved automatically.

In [ ]:
# 0. Imports, reproducibility, and GPU setup
import copy
import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU를 사용할 수 없습니다. GPU 커널을 확인하세요.")

device = torch.device("cuda:0")
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

print(f"사용 디바이스: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 1. Stratified split and street-only cap
dataset_path = "CNN_train_test/CNN_train_test_image"
street_class = "스트리트"
street_cap = 10_000
batch_size = 32
workers = 4

eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(
        degrees=5,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05),
        interpolation=transforms.InterpolationMode.BILINEAR,
        fill=(128, 128, 128),
    ),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

full_dataset = datasets.ImageFolder(dataset_path, transform=eval_transforms)
class_names = full_dataset.classes
num_classes = len(class_names)
targets = np.asarray(full_dataset.targets)

if street_class not in class_names:
    raise ValueError(f"클래스를 찾을 수 없습니다: {street_class}")

street_label = class_names.index(street_class)
train_ids, val_ids = train_test_split(
    np.arange(len(full_dataset)),
    test_size=0.2,
    stratify=targets,
    random_state=SEED,
)

# Only the training split is changed. Validation remains natural and untouched.
rng = np.random.default_rng(SEED)
street_train_ids = train_ids[targets[train_ids] == street_label]
other_train_ids = train_ids[targets[train_ids] != street_label]

if len(street_train_ids) > street_cap:
    street_train_ids = rng.choice(
        street_train_ids,
        size=street_cap,
        replace=False,
    )

capped_train_ids = np.concatenate([other_train_ids, street_train_ids])
rng.shuffle(capped_train_ids)

train_source = copy.copy(full_dataset)
train_source.transform = train_transforms
train_dataset = Subset(train_source, capped_train_ids.tolist())
val_dataset = Subset(full_dataset, val_ids.tolist())

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=workers,
    pin_memory=True,
    persistent_workers=workers > 0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=workers,
    pin_memory=True,
    persistent_workers=workers > 0,
)

all_counts = np.bincount(targets, minlength=num_classes)
train_counts = np.bincount(targets[capped_train_ids], minlength=num_classes)
val_counts = np.bincount(targets[val_ids], minlength=num_classes)
balance_table = pd.DataFrame({
    "class": class_names,
    "all_images": all_counts,
    "train_images_after_cap": train_counts,
    "validation_images": val_counts,
})

print(balance_table.to_string(index=False))
print(f"전체 이미지: {len(full_dataset):,}장")
print(f"V3 학습 이미지: {len(train_dataset):,}장")
print(f"V3 검증 이미지: {len(val_dataset):,}장")
print(f"스트리트 학습 이미지: {train_counts[street_label]:,}장")
print(f"학습 배치: {len(train_loader):,}개/epoch")

In [ ]:
# 2. ConvNeXt-Tiny with the same fine-tuning setup as V2
weights = models.ConvNeXt_Tiny_Weights.DEFAULT
mood_model = models.convnext_tiny(weights=weights)

for param in mood_model.parameters():
    param.requires_grad = False

in_features = mood_model.classifier[2].in_features
mood_model.classifier = nn.Sequential(
    nn.Flatten(1),
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, num_classes),
)

# Same as V2: fine-tune only the final ConvNeXt feature block.
for param in mood_model.features[-1].parameters():
    param.requires_grad = True

mood_model = mood_model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW([
    {"params": mood_model.features[-1].parameters(), "lr": 1e-5},
    {"params": mood_model.classifier.parameters(), "lr": 1e-4},
], weight_decay=1e-4)

amp_enabled = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

trainable_params = sum(
    param.numel() for param in mood_model.parameters() if param.requires_grad
)
print(f"학습 가능한 파라미터: {trainable_params:,}")
print(mood_model)

In [ ]:
# 3. Training, metrics, logging, and best checkpoint
epochs = 100
train_acc_history = []
val_acc_history = []
train_loss_history = []
val_loss_history = []
val_macro_f1_history = []
val_balanced_acc_history = []

model_dir = Path("Model")
model_dir.mkdir(exist_ok=True)
log_path = model_dir / f"v3_street_cap_{street_cap}_training_log.txt"
best_model_path = model_dir / f"fashion_mood_model_pytorch3_street_cap_{street_cap}_best.pth"
last_model_path = model_dir / f"fashion_mood_model_pytorch3_street_cap_{street_cap}_last.pth"
metadata_path = model_dir / f"fashion_mood_model_pytorch3_street_cap_{street_cap}.json"
with log_path.open("a", encoding="utf-8") as log_file:
    log_file.write("\n\n" + "=" * 100 + "\n")
    log_file.write(f"V3 run: street cap = {street_cap:,}, epochs = {epochs}\n")
    log_file.write("=" * 100 + "\n")
best_macro_f1 = -1.0
best_epoch = 0
for epoch in range(epochs):
    mood_model.train()
    for module in list(mood_model.features.children())[:-1]:
        module.eval()
    mood_model.features[-1].train()
    mood_model.classifier.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{epochs} [Train]",
        unit="batch",
    )
    for inputs, labels in train_bar:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            outputs = mood_model(inputs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * labels.size(0)
        total_train += labels.size(0)
        correct_train += (outputs.argmax(1) == labels).sum().item()
        train_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            acc=f"{correct_train / total_train:.1%}",
        )
    train_loss = running_loss / total_train
    train_acc = correct_train / total_train
    train_loss_history.append(train_loss)
    train_acc_history.append(train_acc)
    mood_model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    actual_labels = []
    predicted_labels = []
    val_bar = tqdm(
        val_loader,
        desc=f"Epoch {epoch + 1}/{epochs} [Valid]",
        unit="batch",
    )
    with torch.inference_mode():
        for inputs, labels in val_bar:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=amp_enabled,
            ):
                outputs = mood_model(inputs)
                loss = criterion(outputs, labels)
            predicted = outputs.argmax(1)
            val_running_loss += loss.item() * labels.size(0)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            actual_labels.extend(labels.cpu().tolist())
            predicted_labels.extend(predicted.cpu().tolist())
    val_loss = val_running_loss / total_val
    val_acc = correct_val / total_val
    val_macro_f1 = f1_score(
        actual_labels,
        predicted_labels,
        labels=list(range(num_classes)),
        average="macro",
        zero_division=0,
    )
    val_balanced_acc = balanced_accuracy_score(
        actual_labels,
        predicted_labels,
    )

    val_loss_history.append(val_loss)
    val_acc_history.append(val_acc)
    val_macro_f1_history.append(val_macro_f1)
    val_balanced_acc_history.append(val_balanced_acc)

    log_line = (
        f"Epoch [{epoch + 1}/{epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2%} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2%} | "
        f"Val Macro F1: {val_macro_f1:.4f} | "
        f"Val Balanced Acc: {val_balanced_acc:.2%}"
    )
    print(log_line, flush=True)
    with log_path.open("a", encoding="utf-8") as log_file:
        log_file.write(log_line + "\n")
    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        best_epoch = epoch + 1
        torch.save(mood_model.state_dict(), best_model_path)
        print(f"최고 Macro F1 모델 저장: {best_model_path}")
torch.save(mood_model.state_dict(), last_model_path)
metadata_path.write_text(
    json.dumps({
        "class_names": class_names,
        "street_class": street_class,
        "street_cap": street_cap,
        "seed": SEED,
        "epochs_completed": epochs,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_macro_f1,
        "sampling": "all non-street training images plus capped street images",
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("V3 학습 완료!")
print(f"최고 epoch: {best_epoch}")
print(f"최고 Val Macro F1: {best_macro_f1:.4f}")

In [ ]:
# 4. Plot the V3 metrics
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_axis = range(1, len(train_loss_history) + 1)

axes[0].plot(epochs_axis, train_loss_history, label="Train Loss")
axes[0].plot(epochs_axis, val_loss_history, label="Val Loss")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_axis, train_acc_history, label="Train Acc")
axes[1].plot(epochs_axis, val_acc_history, label="Val Acc")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(epochs_axis, val_macro_f1_history, label="Val Macro F1")
axes[2].plot(epochs_axis, val_balanced_acc_history, label="Val Balanced Acc")
axes[2].set_title("Class-balanced metrics")
axes[2].set_xlabel("Epoch")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()
best_idx = int(np.argmax(val_macro_f1_history))
print(f"최고 Macro F1 epoch: {best_idx + 1}")
print(f"최고 Val Macro F1: {val_macro_f1_history[best_idx]:.4f}")
print(f"해당 Val Acc: {val_acc_history[best_idx]:.2%}")
print(f"해당 Val Balanced Acc: {val_balanced_acc_history[best_idx]:.2%}")